In [ ]:
"""
Per-group spectral (PSD) budget on the native Summit data.

Decides whether multi-rate sampling is needed: for each decoder group it reports
the fraction of variance ABOVE a candidate decimation rate's Nyquist. A large value
(esp. for V_flow_sec / the G_Vs group) means that rate aliases the channel and a
finer rate is worth it; a small value means a single coarse rate is fine.
"""
import sys, os
sys.path.insert(0, os.path.abspath("../../"))   # repo root (has fmu2ml/)

import pandas as pd
import fmu2ml.surrogate as surrogate
from fmu2ml.surrogate.data import store
from fmu2ml.surrogate.evaluate.spectral import per_group_psd_budget, print_psd_budget

#  settings ───────────────────────────────────────────────────────────────
DATA_DIR        = "../../../summit/data/systematic/"   # parent of chunk_<id>/
N_CHUNKS        = 10            # chunks to sample for the PSD (1 is usually enough)
CANDIDATE_RATES = [1,2,3, 5, 10, 30, 60]
DT              = 1.0          # native sample period (s); 1.0 for 1 Hz data
# Use the SAME config you train Phase 5/6 with, so column groups match the data.
config = surrogate.FederatedConfig(system_name="summit", num_cdus=257)


In [ ]:
#  discover chunks ─────────────────────────────────────────────────────────
files = store.discover_chunk_files(DATA_DIR)          # [(chunk_id, parquet_path)]
if not files:
    raise SystemExit(f"No chunk_*/fmu_output_*_operational.parquet under {DATA_DIR!r}")
print(f"Found {len(files)} chunks; sampling {min(N_CHUNKS, len(files))} for the PSD.")


Found 16 chunks; sampling 10 for the PSD.


In [ ]:
#  column_info from one chunk (names only; cheap) ──────────────────────────
first_df = pd.read_parquet(files[0][1])
column_info = surrogate.build_federated_column_info(first_df, config, verbose=False)
dynamic_cols = column_info["dynamic_cols"]
if not column_info.get("temp_indices"):
    raise SystemExit(
        "column groups are empty — the config's output_patterns don't match the data. "
        "Use the exact FederatedConfig from your phase5/phase6 notebook.")
del first_df


In [ ]:
#  read dynamic columns from a few chunks (one at a time) ──────────────────
dynamic_chunks = []
for cid, path in files[:N_CHUNKS]:
    df = pd.read_parquet(path, columns=dynamic_cols)   # column pushdown; native 1 Hz order
    dynamic_chunks.append(df[dynamic_cols].to_numpy("float32"))
    del df
    print(f"  chunk_{cid}: {dynamic_chunks[-1].shape[0]:,} native rows")


  chunk_0: 29,183 native rows
  chunk_1: 29,184 native rows
  chunk_2: 29,183 native rows
  chunk_3: 29,184 native rows
  chunk_4: 29,183 native rows
  chunk_5: 29,184 native rows
  chunk_6: 29,183 native rows
  chunk_7: 29,184 native rows
  chunk_8: 29,183 native rows
  chunk_9: 29,184 native rows


In [ ]:
#  PSD budget ──────────────────────────────────────────────────────────────
report = per_group_psd_budget(dynamic_chunks, dynamic_cols, column_info,
                              candidate_rates=CANDIDATE_RATES, dt=DT)
print_psd_budget(report)

print("\nRead: a high 'mean/max %' at a rate means that rate ALIASES the group "
      "(needs a finer rate). Low % across rates means a single coarse rate is safe.")


Per-group spectral budget (native fs = 1 Hz)
fraction of variance above a candidate rate's Nyquist (higher = more aliasing):
  group T:
    rate   1 (1s, f_nyq=0.5 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[1].cdu[1].summary.T_prim_s)
    rate   2 (2s, f_nyq=0.25 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[14].cdu[1].summary.T_sec_r)
    rate   3 (3s, f_nyq=0.1667 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[14].cdu[1].summary.T_sec_r)
    rate   5 (5s, f_nyq=0.1 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[14].cdu[1].summary.T_sec_r)
    rate  10 (10s, f_nyq=0.05 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[14].cdu[1].summary.T_sec_r)
    rate  30 (30s, f_nyq=0.01667 Hz): mean=  0.0%  max=  0.0% (worst: simulator[1].datacenter[1].computeBlock[14].cdu[1].summary.T_sec_r)
    rate  60 (60s, f_nyq=0.008333 Hz): mean=  0.0%  ma